In [44]:
import numpy as np
import pandas as pd

Age

In [45]:
#Load files

#PATH_CENSUS_AGE = "path/to/census_age_variables.csv"
PATH_CENSUS_AGE = "../data/raw/census/age.csv"

age = pd.read_csv(PATH_CENSUS_AGE)

In [46]:
# Rename columns and restrict to England

age = age.rename(columns={
    "2023 Upper tier local authorities Code": "area_code",
    "2023 Upper tier local authorities": "area_name",
    "Age (B) (7 categories) Code": "age_code",
    "Age (B) (7 categories)": "age_group",
    "Observation": "count"
})

age["count"] = pd.to_numeric(
    age["count"],
    errors="coerce"
)


age = age.loc[
    age["area_code"].str.startswith("E")
].copy()


In [47]:
# Filter to 65+ 

age_map = {
    "Aged 65 to 69 years": "age_65_69",
    "Aged 70 to 74 years": "age_70_74",
    "Aged 75 to 79 years": "age_75_79",
    "Aged 80 to 84 years": "age_80_84",
    "Aged 85 to 89 years": "age_85_plus",
    "Aged 90 years and over": "age_85_plus"
}

age["age_constraint"] = age["age_group"].map(age_map)

age65 =age.loc[
    age["age_constraint"].notna()
].copy()

age65 = (
   age65.groupby(
        [
            "area_code",
            "area_name",
            "age_constraint"
        ],
        as_index=False
    )["count"]
    .sum()
)

In [48]:
# Calculate totals
total_population =age.groupby("area_code")["count"].sum()

age65["population_65plus"] =age65.groupby("area_code")["count"].transform("sum")
age65["total_population"] =age65["area_code"].map(total_population)
age65["rate"] =age65["count"] /age65["population_65plus"]

age_constraint = (
   age65.pivot_table(
        index=["area_code", "area_name", "population_65plus", "total_population"],
        columns="age_constraint",
        values="rate",
    )
    .reset_index()
)
age_constraint.columns.name = None

age_constraint[["area_code", "age_65_69", "age_70_74", "age_75_79", "age_80_84", "age_85_plus"]].to_csv("../data/constraints/age.csv")
age_constraint.head()

,area_code,area_name,population_65plus,total_population,age_65_69,age_70_74,age_75_79,age_80_84,age_85_plus
0,E06000001,Hartlepool,18233,92339,0.287775,0.278835,0.176713,0.135085,0.121593
1,E06000002,Middlesbrough,24180,143926,0.298098,0.267328,0.177833,0.133830,0.122911
2,E06000003,Redcar and Cleveland,31747,136531,0.267647,0.273254,0.199735,0.141084,0.118279
3,E06000004,Stockton-on-Tees,37184,196595,0.290259,0.275737,0.184811,0.133229,0.115964
4,E06000005,Darlington,22040,107799,0.270327,0.271325,0.188249,0.141606,0.128494


In [49]:
age_constraint[["area_code", "area_name", "population_65plus", "total_population"]].to_csv("../data/constraints/total.csv")

Sex

In [50]:
#PATH_CENSUS_SEX = "path/to/census_sex_variables.csv"
PATH_CENSUS_SEX = "../data/raw/census/sex.csv"

sex = pd.read_csv(PATH_CENSUS_SEX)

In [51]:
# Rename columns and restrict to England

sex = sex.rename(columns={
    "2023 Upper tier local authorities Code": "area_code",
    "2023 Upper tier local authorities": "area_name",
    "Age (3 categories) Code": "age_code",
    "Age (3 categories)": "age_group",
    "Sex (2 categories) Code": "sex_code",
    "Sex (2 categories)": "sex",
    "Observation": "count"
})

sex["count"] = pd.to_numeric(
    sex["count"],
    errors="coerce"
)


sex = sex.loc[
    sex["area_code"].str.startswith("E")
].copy()

sex.head()

,area_code,area_name,age_code,age_group,sex_code,sex,count
0,E06000001,Hartlepool,1,Aged 15 years and under,1,Female,8589
1,E06000001,Hartlepool,1,Aged 15 years and under,2,Male,9063
2,E06000001,Hartlepool,2,Aged 16 to 64 years,1,Female,29158
3,E06000001,Hartlepool,2,Aged 16 to 64 years,2,Male,27295
4,E06000001,Hartlepool,3,Aged 65 years and over,1,Female,9906


In [52]:
# Filter to 65+ 

sex65 = sex[sex["age_code"] == 3].copy()


sex65["count"] = pd.to_numeric(sex65["count"], errors="coerce")

sex65["population_65plus"] = (
    sex65.groupby(["area_code", "area_name"])["count"]
       .transform("sum")
)

In [53]:
# Calculate totals


sex65["rate"] = sex65["count"] / sex65["population_65plus"]


sex_constraints = (
    sex65.pivot_table(
        index=["area_code", "area_name", "population_65plus"],
        columns="sex",
        values="rate"
    )
    .reset_index()
    .rename(columns={
        "Female": "female",
        "Male": "male"
    })
)

sex_constraints.columns.name = None

sex_constraints[["area_code", "female", "male"]].to_csv("../data/constraints/sex.csv")

In [54]:
sex_constraints.head()

,area_code,area_name,population_65plus,female,male
0,E06000001,Hartlepool,18233,0.543301,0.456699
1,E06000002,Middlesbrough,24178,0.545289,0.454711
2,E06000003,Redcar and Cleveland,31747,0.539799,0.460201
3,E06000004,Stockton-on-Tees,37184,0.537973,0.462027
4,E06000005,Darlington,22040,0.538113,0.461887


Ethnicity

In [55]:
#Load files

#PATH_CENSUS_ETHNICITY = "path/to/census_ethnicity_variables.csv"
PATH_CENSUS_ETHNICITY = "../data/raw/census/ethnicity.csv"

ethnicity = pd.read_csv(PATH_CENSUS_ETHNICITY)

In [56]:
# Rename columns and restrict to England

ethnicity = ethnicity.rename(columns={
    "2023 Upper tier local authorities Code": "area_code",
    "2023 Upper tier local authorities": "area_name",
    "Age (3 categories) Code": "age_code",
    "Age (3 categories)": "age_group",
    "Ethnic group (6 categories) Code": "ethnicity_code",
    "Ethnic group (6 categories)": "ethnicity",
    "Observation": "count"
})

ethnicity["count"] = pd.to_numeric(
    ethnicity["count"],
    errors="coerce"
)


ethnicity = ethnicity.loc[
    ethnicity["area_code"].str.startswith("E")
].copy()


In [57]:
# Filter to 65+ 


ethnicity65 = ethnicity[ethnicity["age_code"] == 3].copy()


ethnicity65["count"] = pd.to_numeric(ethnicity65["count"], errors="coerce")

print(ethnicity65["ethnicity"].value_counts(dropna=False))

ethnicity
Does not apply                                             153
Asian, Asian British or Asian Welsh                        153
Black, Black British, Black Welsh, Caribbean or African    153
Mixed or Multiple ethnic groups                            153
White                                                      153
Other ethnic group                                         153
Name: count, dtype: int64


In [58]:
# Calculate totals

ethnicity65["ethnicity_group"] = ethnicity65["ethnicity"].apply(
    lambda x: "ethnicity_white"
    if str(x).strip().lower() == "white"
    else "ethnicity_non_white"
)

ethnicity65 = (
    ethnicity65.groupby(
        ["area_code", "area_name", "ethnicity_group"],
        as_index=False
    )["count"]
    .sum()
)

ethnicity65["population_65plus"] = (
    ethnicity65.groupby(["area_code", "area_name"])["count"]
         .transform("sum")
)

ethnicity65["rate"] = (
    ethnicity65["count"] / ethnicity65["population_65plus"]
)


ethnicity_constraints = (
    ethnicity65.pivot_table(
        index=["area_code", "area_name", "population_65plus"],
        columns="ethnicity_group",
        values="rate"
    )
    .reset_index()
)

ethnicity_constraints.columns.name = None

ethnicity_constraints[["area_code", "ethnicity_non_white", "ethnicity_white"]].to_csv("../data/constraints/ethnicity.csv")

ethnicity_constraints.head()

,area_code,area_name,population_65plus,ethnicity_non_white,ethnicity_white
0,E06000001,Hartlepool,18233,0.008062,0.991938
1,E06000002,Middlesbrough,24178,0.041277,0.958723
2,E06000003,Redcar and Cleveland,31748,0.004914,0.995086
3,E06000004,Stockton-on-Tees,37184,0.019767,0.980233
4,E06000005,Darlington,22040,0.015064,0.984936


Country of birth

In [59]:
#Load files


#PATH_CENSUS_COUNTRY_OF_BIRTH = "path/to/census_country_of_birth_variables.csv"
PATH_CENSUS_COUNTRY_OF_BIRTH = "../data/raw/census/country_of_birth.csv"

country_of_birth = pd.read_csv(PATH_CENSUS_COUNTRY_OF_BIRTH)

In [60]:
# Rename columns and restrict to England

country_of_birth = country_of_birth.rename(columns={
    "2023 Upper tier local authorities Code": "area_code",
    "2023 Upper tier local authorities": "area_name",
    "Age (3 categories) Code": "age_code",
    "Age (3 categories)": "age_group",
    "Country of birth (UK) (3 categories) Code": "cob_code",
    "Country of birth (UK) (3 categories)": "cob",
    "Observation": "count"
})

country_of_birth["count"] = pd.to_numeric(
    country_of_birth["count"],
    errors="coerce"
)


country_of_birth = country_of_birth.loc[
    country_of_birth["area_code"].str.startswith("E")
].copy()

In [61]:
# Filter to 65+


country_of_birth65 = country_of_birth[country_of_birth["age_code"] == 3].copy()

print(country_of_birth65["cob"].value_counts(dropna=False))

cob
Does not apply         153
Born in the UK         153
Born outside the UK    153
Name: count, dtype: int64


In [62]:
# Classify UK-born and non-UK-born

COB_GROUPS = {
    "Born in the UK": "uk_born",
    "Born outside the UK": "non_uk_born",
    "Does not apply": None,
}

country_of_birth65["cob_group"] = country_of_birth65["cob"].map(COB_GROUPS)

In [63]:
# Calculate totals

country_of_birth65 = (
    country_of_birth65.groupby(
        ["area_code", "area_name", "cob_group"],
        as_index=False
    )["count"]
    .sum()
)

country_of_birth65["population_65plus"] = (
    country_of_birth65.groupby(["area_code", "area_name"])["count"]
         .transform("sum")
)

country_of_birth65["rate"] = (
    country_of_birth65["count"] / country_of_birth65["population_65plus"]
)


country_of_birth_constraints = (
    country_of_birth65.pivot_table(
        index=["area_code", "area_name", "population_65plus"],
        columns="cob_group",
        values="rate"
    )
    .reset_index()
)

country_of_birth_constraints.columns.name = None

country_of_birth_constraints[["area_code", "uk_born", "non_uk_born"]].to_csv("../data/constraints/country_of_birth.csv")

country_of_birth_constraints.head()

,area_code,area_name,population_65plus,non_uk_born,uk_born
0,E06000001,Hartlepool,18233,0.019580,0.980420
1,E06000002,Middlesbrough,24178,0.055340,0.944660
2,E06000003,Redcar and Cleveland,31747,0.018238,0.981762
3,E06000004,Stockton-on-Tees,37184,0.033644,0.966356
4,E06000005,Darlington,22040,0.034528,0.965472


Health

In [64]:
#Load files

#PATH_CENSUS_HEALTH = "path/to/census_health_variables.csv"
PATH_CENSUS_HEALTH = "../data/raw/census/health.csv"

health = pd.read_csv(PATH_CENSUS_HEALTH)

In [65]:
# Rename columns and restrict to England

health = health.rename(columns={
    "2023 Upper tier local authorities Code": "area_code",
    "2023 Upper tier local authorities": "area_name",
    "Age (3 categories) Code": "age_code",
    "Age (3 categories)": "age_group",
    "General health (6 categories) Code": "health_code",
    "General health (6 categories)": "health",
    "Observation": "count"
})

# Filter to 65+

health65 = health[health["age_code"] == 3].copy()


health65 = health65.loc[
    health65["area_code"].str.startswith("E")
].copy()

health65["count"] = pd.to_numeric(
    health65["count"],
    errors="coerce"
)

health65.head()

,area_code,area_name,age_code,age_group,health_code,health,count
12,E06000001,Hartlepool,3,Aged 65 years and over,-8,Does not apply,0
13,E06000001,Hartlepool,3,Aged 65 years and over,1,Very good health,2349
14,E06000001,Hartlepool,3,Aged 65 years and over,2,Good health,6458
15,E06000001,Hartlepool,3,Aged 65 years and over,3,Fair health,6317
16,E06000001,Hartlepool,3,Aged 65 years and over,4,Bad health,2350


In [66]:
# Classify health categories

health_map = {
    "Very good health": "health_very_good_health",
    "Good health": "health_good_health",
    "Fair health": "health_fair_health",
    "Bad health": "health_bad_health",
    "Very bad health": "health_bad_health",
}

health65["health_group"] = health65["health"].map(
    health_map
)


In [67]:
# Calculate totals

health65 = (
    health65.groupby(
        [
            "area_code",
            "area_name",
            "health_group"
        ],
        as_index=False
    )["count"]
    .sum()
)


health65["population_65plus"] = (
    health65
    .groupby(
        ["area_code", "area_name"]
    )["count"]
    .transform("sum")
)

health65["rate"] = (
    health65["count"]
    / health65["population_65plus"]
)
health_constraints = (
    health65.pivot_table(
        index=[
            "area_code",
            "area_name",
            "population_65plus"
        ],
        columns="health_group",
        values="rate"
    )
    .reset_index()
)

health65.columns.name = None
health_constraints[["area_code", "health_bad_health", "health_fair_health", "health_good_health", "health_very_good_health"]].to_csv("../data/constraints/health.csv")

health_constraints.head()


health_group,area_code,area_name,population_65plus,health_bad_health,health_fair_health,health_good_health,health_very_good_health
0,E06000001,Hartlepool,18233,0.170515,0.346460,0.354193,0.128832
1,E06000002,Middlesbrough,24178,0.177434,0.312350,0.367855,0.142361
2,E06000003,Redcar and Cleveland,31747,0.153495,0.320566,0.381453,0.144486
3,E06000004,Stockton-on-Tees,37184,0.143933,0.311666,0.389576,0.154825
4,E06000005,Darlington,22040,0.133122,0.307169,0.401679,0.158031


NS-SEC

In [68]:
#Load files

#PATH_CENSUS_NSSEC = "path/to/census_nssec_variables.csv"
PATH_CENSUS_NSSEC = "../data/raw/census/nssec.csv"

nssec = pd.read_csv(PATH_CENSUS_NSSEC)



In [69]:
# Rename columns and restrict to England

nssec = nssec.rename(columns={
    "2023 Upper tier local authorities Code": "area_code",
    "2023 Upper tier local authorities": "area_name",
    "Age (3 categories) Code": "age_code",
    "Age (3 categories)": "age_group",
    "National Statistics Socio-economic Classification (NS-SeC) (10 categories) Code": "nssec_code",
    "National Statistics Socio-economic Classification (NS-SeC) (10 categories)": "nssec",
    "Observation": "count"
})

nssec["nssec_code"] = pd.to_numeric(nssec["nssec_code"], errors="raise").astype(int)


nssec = nssec[nssec["area_code"].str.startswith("E")].copy()

nssec.head()


,area_code,area_name,age_code,age_group,nssec_code,nssec,count
0,E06000001,Hartlepool,1,Aged 15 years and under,-8,Does not apply,17652
1,E06000001,Hartlepool,1,Aged 15 years and under,1,"L1, L2 and L3: Higher managerial, administrati...",0
2,E06000001,Hartlepool,1,Aged 15 years and under,2,"L4, L5 and L6: Lower managerial, administrativ...",0
3,E06000001,Hartlepool,1,Aged 15 years and under,3,L7: Intermediate occupations,0
4,E06000001,Hartlepool,1,Aged 15 years and under,4,L8 and L9: Small employers and own account wor...,0


In [70]:
# Filter to 65+

nssec65 = nssec[nssec["age_code"] == 3].copy()


In [71]:
# Calculate totals


nssec_map = {1: "nssec_managerial",  2: "nssec_managerial",
             3: "nssec_intermediate", 4: "nssec_intermediate",
             5: "nssec_manual", 6: "nssec_manual", 7: "nssec_manual"}

nssec65["nssec_group"] = nssec65["nssec_code"].map(nssec_map)

nssec65 = nssec65.dropna(subset=["nssec_group"])

nssec65 = (
    nssec65.groupby(
        ["area_code", "area_name", "nssec_group"],
        as_index=False
    )["count"]
    .sum()
)

nssec65["population_65plus"] = (
    nssec65.groupby(["area_code", "area_name"])["count"]
         .transform("sum")
)

nssec65["rate"] = (
    nssec65["count"] / nssec65["population_65plus"]
)


nssec_constraints = (
    nssec65.pivot_table(
        index=["area_code", "area_name", "population_65plus"],
        columns="nssec_group",
        values="rate"
    )
    .reset_index()
)

nssec_constraints.columns.name = None

nssec_constraints[["area_code", "nssec_managerial", "nssec_intermediate", "nssec_manual"]].to_csv("../data/constraints/nssec.csv")

nssec_constraints.head()

,area_code,area_name,population_65plus,nssec_intermediate,nssec_managerial,nssec_manual
0,E06000001,Hartlepool,14116,0.203670,0.259068,0.537263
1,E06000002,Middlesbrough,18689,0.211782,0.270105,0.518112
2,E06000003,Redcar and Cleveland,25723,0.210201,0.290363,0.499436
3,E06000004,Stockton-on-Tees,30373,0.223982,0.320350,0.455668
4,E06000005,Darlington,18089,0.252253,0.327050,0.420698


Partnership status

In [72]:
#Load files

#PATH_CENSUS_PARTNERSHIP = "path/to/census_partnership_variables.csv"
PATH_CENSUS_PARTNERSHIP = "../data/raw/census/partnership.csv"

partnership = pd.read_csv(PATH_CENSUS_PARTNERSHIP)



In [73]:
# Rename columns and restrict to England

partnership = partnership.rename(columns={
    "2023 Upper tier local authorities Code": "area_code",
    "2023 Upper tier local authorities": "area_name",
    "Age (3 categories) Code": "age_code",
    "Age (3 categories)": "age_group",
    "Marital and civil partnership status (3 categories) Code": "partner_code",
    "Marital and civil partnership status (3 categories)": "partner_status",
    "Observation": "count"
})

partnership["count"] = pd.to_numeric(
    partnership["count"],
    errors="coerce"
)

partnership = partnership.loc[
    partnership["area_code"].str.startswith("E")
].copy()

In [74]:
# Filter to 65+

partnership65 = partnership[partnership["age_code"] == 3].copy()


In [75]:
# Calculate totals


partner_map = {
    1: "partner_has_partner",
    2: "partner_does_not_have_partner"
}
partnership65["partner_group"] = (
    partnership65["partner_code"]
    .map(partner_map)
)


partnership65["population_65plus"] = (
    partnership65.groupby(["area_code", "area_name"])["count"]
         .transform("sum")

)


partnership65["rate"] = (
    partnership65["count"]
    / partnership65["population_65plus"]
)


partner_constraint = (
    partnership65
    .pivot_table(
        index=[
            "area_code",
            "area_name",
            "population_65plus"
        ],
        columns="partner_group",
        values="rate"
    )
    .reset_index()
)

partner_constraint[["area_code", "partner_has_partner", "partner_does_not_have_partner"]].to_csv("../data/constraints/partnership.csv")

partner_constraint.head()

partner_group,area_code,area_name,population_65plus,partner_does_not_have_partner,partner_has_partner
0,E06000001,Hartlepool,18233,0.444140,0.555860
1,E06000002,Middlesbrough,24178,0.471875,0.528125
2,E06000003,Redcar and Cleveland,31748,0.427082,0.572918
3,E06000004,Stockton-on-Tees,37184,0.422520,0.577480
4,E06000005,Darlington,22040,0.438022,0.561978
